# Controllable Multi-Axis Music Search — Colab Training

Runs the ingest + head-training pipeline from `music-similarity-mle-plan.md`
on Google Colab instead of the Cornell cluster it was originally scoped for.

**Why this notebook looks different from a typical training notebook:** Colab's
GPU runtimes only have ~64GB of *ephemeral* local disk (wiped every time the
runtime recycles), and the full ingest plan calls for separating 25-30k tracks
into 4 stems each — that's several hundred GB of uncompressed WAV if stored
naively. So this pipeline never holds more than one track's stems on disk at a
time: separate → embed → delete, immediately, with only the small embedding
vectors persisted to Google Drive. See `pipeline/colab_pipeline.py` for the
full rationale.

**Before you start:** Runtime → Change runtime type → GPU (T4 to start; try L4
if it's offered and your budget allows). Then run the cells top to bottom —
every long-running cell is checkpointed to Drive, so if Colab disconnects you
just re-run the same cell and it picks up where it left off.

## 1. Environment check + install

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU -- go to Runtime > Change runtime type > GPU before continuing.")


In [ ]:
# Colab already ships a CUDA-matched torch/torchaudio; requirements-colab.txt
# deliberately excludes them so this install doesn't clobber that wiring.
# Works whether you run it before or after mounting Drive: falls back to the
# Drive copy if the relative path isn't found (Colab's cwd is /content).
from pathlib import Path
req = Path("requirements-colab.txt")
if not req.exists():
    req = Path("/content/drive/MyDrive/music-similarity/requirements-colab.txt")
    assert req.exists(), "Mount Drive first (Section 2), then re-run this cell -- or fix the path above"
!pip install -q -r "{req}"


## 2. Mount Google Drive

Do this **before** adding anything under `/content/drive` to `sys.path` --
the path doesn't exist until this cell finishes, so importing project code
from Drive earlier than this will fail with `ModuleNotFoundError`.

When the auth popup appears, just pick the same account you're running Colab on.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Point Python at the project code

Edit `PROJECT_DIR` below to match exactly where you put the `music-similarity`
folder in your Drive (case-sensitive; it's `MyDrive`, not `My Drive`).

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/music-similarity")  # <-- edit if yours differs
assert PROJECT_DIR.exists(), f"{PROJECT_DIR} not found -- check the path and that Drive finished mounting"
sys.path.insert(0, str(PROJECT_DIR))


## 4. Point the pipeline at persistent storage

In [ ]:
import os

# STORAGE MODE (single account, 15GB Drive) -- pick one:
# (A) DEFAULT: everything on Drive (~10.5GB total: audio crops + embeddings).
#     Survives runtime recycles; needs ~11GB free on Drive. Nothing to set.
# (B) Tight on Drive space? Keep bulky audio on ephemeral local disk instead
#     (~0.6GB on Drive; audio re-downloads in ~30-60 min after a recycle):
# os.environ["MS_AUDIO_DIR"] = "/content/audio"
# Full-song archiving is OFF (no benefit -- Jamendo's CDN is the archive).
os.environ.pop("MS_KEEP_FULL_AUDIO_DIR", None)

from pipeline.config import configure_for_colab
from pipeline.colab_pipeline import gpu_report

# Repoints pipeline.config.PATHS / QDRANT at <PROJECT_DIR>/colab_run so
# embeddings, checkpoints, and the local Qdrant index all persist across
# runtime recycles instead of living on ephemeral local disk.
drive_root = PROJECT_DIR / "colab_run"
configure_for_colab(drive_root)
print(f"Data root: {drive_root}")
print(gpu_report())


## 5. Size your subset to your compute budget

Colab bills GPU time in compute units (CU); rates as of mid-2026 are roughly
1.76 CU/hr on a T4 and 15 CU/hr on an A100, with L4 in between. This cell
gives a rough sizing estimate — treat it as a planning number, not a
guarantee, since actual per-track time depends on track length and whatever
else is running on the shared GPU.

In [ ]:
from pipeline.colab_pipeline import detect_gpu, estimate_hours, estimate_feasible_tracks

COMPUTE_UNITS_BUDGET = 100  # set to what you actually have

gpu = detect_gpu()
print("Detected GPU:", gpu)
hours = estimate_hours(COMPUTE_UNITS_BUDGET, gpu) if gpu else None
tracks = estimate_feasible_tracks(COMPUTE_UNITS_BUDGET, gpu) if gpu else None
if hours:
    print(f"~{hours:.1f} GPU-hours available on this GPU for {COMPUTE_UNITS_BUDGET} CU")
    print(f"~{tracks} tracks feasible through Demucs+MERT (leaves headroom for Phase 2 training + retries)")
else:
    print("Unrecognized/no GPU -- pick a subset size conservatively (a few thousand tracks) and watch actual timing on the first batch instead.")


## 6. Phase 0 — walking skeleton (whole-mix MERT + FAISS)

De-risks the rest of the notebook on a tiny sample before committing GPU-hours to the full ingest.

In [ ]:
# [CPU-OK] download only -- run on a free CPU runtime
from pipeline.download_jamendo import fetch_metadata, read_track_ids, stratified_subset, download_audio
from pipeline.config import PATHS

metadata_path = fetch_metadata("autotagging_top50tags", PATHS.metadata_dir / "jamendo.tsv")
all_rows = read_track_ids(metadata_path)
baseline_rows = stratified_subset(all_rows, n=200, seed=0)
download_audio(baseline_rows, PATHS.audio_dir)
print(f"Downloaded {len(baseline_rows)} tracks for the baseline skeleton")


In [ ]:
# [GPU] embeds 200 tracks with MERT -- run on the GPU session
from pipeline.baseline_faiss import build_index, query_index
from pipeline.config import PATHS

baseline_index_path = PATHS.embeddings_dir / "baseline.index"
build_index(PATHS.audio_dir, baseline_index_path)

# quick sanity listen -- pick any downloaded track id
sample_track_id = baseline_rows[0]["TRACK_ID"].replace("track_", "")
for track_id, score in query_index(baseline_index_path, sample_track_id, k=5):
    print(f"{track_id}\t{score:.3f}")


## 7. Phase 1 — streaming ingest (main event)

Downloads and processes tracks in a loop that separates → embeds → deletes
per track, appending to a Drive-backed JSONL manifest after every track. Safe
to interrupt: re-running this cell skips whatever's already in the manifest.

In [ ]:
# [CPU-OK] download only -- run on a free CPU runtime
N_TRACKS = 5000  # adjust per the budget estimate above

subset_rows = stratified_subset(all_rows, n=N_TRACKS, seed=1)
subset_dir = PATHS.audio_dir / "ingest_subset"
subset_dir.mkdir(parents=True, exist_ok=True)
download_audio(subset_rows, subset_dir)

audio_paths = sorted(subset_dir.glob("*.mp3"))
print(f"{len(audio_paths)} tracks staged for streaming ingest")


## 7a. GPU smoke test

Validates timing and pair-ingest correctness on a tiny sample before the
full run. Everything it writes counts toward the real run (manifests are
resumable), so nothing is wasted.

In [ ]:
# [GPU] Smoke test: timing sample + pair-ingest correctness asserts.
import json, time
from pipeline.colab_pipeline import run_streaming_ingest, run_streaming_pair_ingest, check_disk_headroom, detect_gpu
from pipeline.config import PATHS
from pipeline.download_jamendo import download_audio, stratified_subset

check_disk_headroom(min_free_gb=5.0)
print(f"[SMOKE] GPU: {detect_gpu()}")
manifest_path = PATHS.embeddings_dir / "manifest.jsonl"

# 1) timed 25-track ingest sample (counts toward the full run)
t0 = time.time()
run_streaming_ingest(audio_paths[:25], manifest_path)
per_track = (time.time() - t0) / 25
n_rows = sum(1 for _ in open(manifest_path))
print(f"[SMOKE] ingest: {per_track:.1f}s/track  |  manifest rows so far: {n_rows}")
print(f"[SMOKE] full-run projection: {len(audio_paths)} tracks -> ~{per_track*len(audio_paths)/3600:.1f} GPU-hours")

# 2) 3-track pair ingest + invariance asserts
anchor_manifest = PATHS.embeddings_dir / "pair_anchor.jsonl"
positive_manifest = PATHS.embeddings_dir / "pair_positive.jsonl"
t0 = time.time()
# 3 pair tracks in two-segment mode (cross-section positives)
smoke_pair_dir = PATHS.audio_dir / "pair_subset"
smoke_pair_dir.mkdir(parents=True, exist_ok=True)
download_audio(stratified_subset(all_rows, n=3, seed=2), smoke_pair_dir, two_segments=True)
smoke_pair_paths = sorted(smoke_pair_dir.glob("*__a.mp3"))[:3]
run_streaming_pair_ingest(smoke_pair_paths, anchor_manifest, positive_manifest)
print(f"[SMOKE] pair ingest: {(time.time()-t0)/3:.1f}s/track")

def _read_manifest(path):
    rows = {}
    for l in open(path):
        try:
            r = json.loads(l)
            rows[r["track_id"]] = r
        except (json.JSONDecodeError, KeyError):
            continue  # partial write from a crashed session; clean copy exists later
    return rows

a_rows = _read_manifest(anchor_manifest)
p_rows = _read_manifest(positive_manifest)
shared = set(a_rows) & set(p_rows)
assert len(shared) >= 3, f"only {len(shared)} complete pairs -- pair ingest is failing"
for tid in shared:
    for stem, emb in a_rows[tid]["embeddings"].items():
        pe = p_rows[tid]["embeddings"].get(stem)
        assert pe is None or emb != pe, f"positive identical to anchor for {tid}/{stem}"
print(f"[SMOKE] {len(shared)} complete pairs; anchor != positive for every shared stem -- OK")
print("[SMOKE] stems present:", {t: sorted(a_rows[t]["embeddings"]) for t in list(shared)[:3]})
print("[SMOKE] ALL CHECKS PASSED")


In [ ]:
# [GPU]
from pipeline.colab_pipeline import run_streaming_ingest, check_disk_headroom
from pipeline.config import PATHS

check_disk_headroom(min_free_gb=5.0)

manifest_path = PATHS.embeddings_dir / "manifest.jsonl"  # lives under drive_root/data -> persists
run_streaming_ingest(audio_paths, manifest_path)


In [ ]:
from pipeline.colab_pipeline import manifest_to_parquet
from pipeline.config import PATHS

raw_embeddings_parquet = PATHS.embeddings_dir / "mert_embeddings.parquet"
manifest_to_parquet(manifest_path, raw_embeddings_parquet)


## 8. Phase 2 — aspect head training

Reserves a smaller sub-subset for pair (anchor/positive) ingest, since this
pass runs Demucs *and* the invariance augmentations *and* two embedding calls
per stem — roughly double the compute of Phase 1's ingest per track.

In [ ]:
# [CPU-OK] Phase 2 pair-subset download (network only)
N_PAIR_TRACKS = 1500  # smaller than N_TRACKS -- Phase 2 is compute-heavier per track

pair_rows = stratified_subset(all_rows, n=N_PAIR_TRACKS, seed=2)
pair_dir = PATHS.audio_dir / "pair_subset"
pair_dir.mkdir(parents=True, exist_ok=True)
# two disjoint 30s segments per track -> cross-section positives for
# drums/vocals/timbre (stronger than sub-windows of one clip)
download_audio(pair_rows, pair_dir, two_segments=True)
pair_audio_paths = sorted(pair_dir.glob("*__a.mp3"))
print(f"{len(pair_audio_paths)} pair tracks staged")


In [ ]:
# [GPU] Phase 2 pair ingest -- do NOT run on CPU (Demucs on CPU is brutally slow)
from pipeline.colab_pipeline import run_streaming_pair_ingest, manifest_to_parquet

anchor_manifest = PATHS.embeddings_dir / "pair_anchor.jsonl"
positive_manifest = PATHS.embeddings_dir / "pair_positive.jsonl"
run_streaming_pair_ingest(pair_audio_paths, anchor_manifest, positive_manifest)

anchor_parquet = manifest_to_parquet(anchor_manifest, PATHS.embeddings_dir / "pair_anchor.parquet")
positive_parquet = manifest_to_parquet(positive_manifest, PATHS.embeddings_dir / "pair_positive.parquet")


In [ ]:
# [GPU]
from training.train import train_one_aspect

trained_heads = {}
for aspect in ["rhythm", "melody", "timbre", "vocal"]:
    print(f"--- training {aspect} head ---")
    trained_heads[aspect] = train_one_aspect(
        aspect=aspect,
        anchor_path=anchor_parquet,
        positive_path=positive_parquet,
        loss_name="info_nce",
        epochs=15,  # short run to fit a Colab session; raise once you've validated the loop
    )


Checkpoints land in `<drive_root>/data/checkpoints/<aspect>_head.pt` — persisted, so later sessions can `torch.load` them directly instead of retraining.

## 9. Eval

Triplet accuracy needs a triplet pool; the quickest path on Colab is generating invariance-style triplets from the pair manifests you already have (real MoisesDB/MUSDB triplets are the paper track's job, run separately once those corpora are downloaded).

In [ ]:
import json
import numpy as np
import pandas as pd

from eval.precision_at_k import build_tag_index, precision_at_k

# Precision@k against Jamendo tags, using the raw (untrained) per-stem MERT
# embeddings from Phase 1 as a quick sanity baseline before trusting the
# trained heads' numbers.
df = pd.read_parquet(raw_embeddings_parquet)
drum_only = df[df["stem"] == "drums"]
embeddings = {row["track_id"]: np.asarray(row["embedding"]) for _, row in drum_only.iterrows()}
tag_index = build_tag_index(metadata_path)

p_at_10 = precision_at_k(embeddings, tag_index, k=10, n_queries=200)
print(f"Baseline (raw drum-stem MERT) precision@10: {p_at_10:.4f}")


## 10. Index into local Qdrant + demo query

Uses Qdrant's embedded local mode (`QdrantClient(path=...)`) -- no server process needed, and it's already pointed at Drive by `setup_colab` above.

In [ ]:
from training.aspect_heads import MultiAspectModel
import torch

model = MultiAspectModel()
for aspect, head in model.heads.items():
    ckpt = PATHS.checkpoints_dir / f"{aspect}_head.pt"
    if ckpt.exists():
        head.load_state_dict(torch.load(ckpt, map_location="cpu"))
        head.eval()

# Project every track in the Phase-1 raw-embeddings parquet through the
# trained heads to build the aspect-vector table qdrant_index.py expects.
rows = []
for track_id, group in df.groupby("track_id"):
    stem_embeddings = {
        row["stem"]: torch.tensor(row["embedding"], dtype=torch.float32).unsqueeze(0) for _, row in group.iterrows()
    }
    with torch.no_grad():
        projected = model.project_all(stem_embeddings)
    row = {"track_id": track_id}
    for aspect, vec in projected.items():
        row[aspect] = vec.squeeze(0).numpy().tolist()
    rows.append(row)

aspect_df = pd.DataFrame(rows)
aspect_parquet = PATHS.embeddings_dir / "aspect_vectors.parquet"
aspect_df.to_parquet(aspect_parquet)
print(f"Projected {len(aspect_df)} tracks through trained heads")


In [ ]:
from pipeline.qdrant_index import get_client, ensure_collection, load_aspect_table, upsert_tracks

client = get_client()
ensure_collection(client)
table = load_aspect_table(aspect_parquet)
upsert_tracks(client, table)
print(f"Indexed {len(table)} tracks into local Qdrant at {drive_root / 'qdrant_local'}")


In [ ]:
from api.search import weighted_fusion_search, fetch_seed_vectors

seed_id = table.iloc[0]["track_id"]
seed_vectors = fetch_seed_vectors(client, seed_id)
weights = {"rhythm": 0.4, "melody": 0.2, "timbre": 0.2, "vocal": 0.1, "lyric": 0.1}
results, latency_ms = weighted_fusion_search(client, seed_vectors, weights, top_k=10, exclude_id=seed_id)

print(f"Seed: {seed_id}  (latency {latency_ms:.1f} ms)")
for r in results:
    print(f"  {r['track_id']}\tscore={r['score']:.3f}")


## 11. Lyric axis + playable audio (re-index)

Transcribes vocals with faster-whisper (instrumentals skipped via an RMS
gate -- expect a large fraction of Jamendo to be instrumental), embeds the
lyrics, and re-upserts every track with its lyric vector plus the CDN `path`
so the web app can stream audio. Budget: ~4-6s/track on a T4 => ~5-7
GPU-hours for the full subset. Resumable like every other ingest.

In [ ]:
# [GPU] streaming lyrics pass (Demucs vocals -> RMS gate -> whisper)
from pipeline.colab_pipeline import run_streaming_lyrics, lyrics_manifest_to_vectors
from pipeline.config import PATHS

lyrics_manifest = PATHS.lyrics_dir / "lyrics_manifest.jsonl"
run_streaming_lyrics(audio_paths, lyrics_manifest)
lyric_parquet = lyrics_manifest_to_vectors(lyrics_manifest, PATHS.lyrics_dir / "lyric_vectors.parquet")


In [ ]:
# merge lyric vectors + CDN paths into the aspect table, re-upsert
import pandas as pd
from pipeline.download_jamendo import read_track_ids
from pipeline.qdrant_index import get_client, upsert_tracks

aspect_df = pd.read_parquet(aspect_parquet)
lyr = pd.read_parquet(lyric_parquet)
aspect_df = aspect_df.drop(columns=[c for c in ("lyric", "path") if c in aspect_df]).merge(lyr, on="track_id", how="left")
path_map = {r["TRACK_ID"].replace("track_", ""): r.get("PATH") for r in read_track_ids(metadata_path)}
aspect_df["path"] = aspect_df["track_id"].map(path_map)
aspect_df.to_parquet(aspect_parquet)

client = get_client()
upsert_tracks(client, aspect_df)
n_lyr = aspect_df["lyric"].notna().sum()
print(f"Re-indexed {len(aspect_df)} tracks; {n_lyr} have lyric vectors ({n_lyr/len(aspect_df):.0%})")


In [ ]:
# demo: same seed, lyric-weighted query now works
from api.search import weighted_fusion_search, fetch_seed_vectors

seed_vectors = fetch_seed_vectors(client, seed_id)
weights = {"lyric": 0.6, "melody": 0.1, "timbre": 0.1, "rhythm": 0.1, "vocal": 0.1}
results, latency_ms = weighted_fusion_search(client, seed_vectors, weights, top_k=10, exclude_id=seed_id)
print(f"Lyric-weighted results for seed {seed_id} ({latency_ms:.1f} ms):")
for r in results:
    print(f"  {r['track_id']}\tscore={r['score']:.3f}\tpath={r.get('path')}")


## 12. Paper track — triplet benchmark: embed + machine baselines

Embeds the rendered triplet pool (`training/triplet_engine.py` + `training/triplet_render.py`, run on a laptop against the MUSDB18-HQ/MoisesDB zips -- real-audio stem manipulation is CPU-only, no GPU needed for that half) through Demucs + MERT + the trained AspectHeads, then scores triplet accuracy. This is the GPU-bound half.

Before running, upload to Drive under `<drive_root>/triplets/`:
- `pool_pilot.jsonl` (the recipe pool, from the laptop's `data/triplets/pool_pilot.jsonl`)
- `audio_pilot/` (rendered `<triplet_id>/{anchor,positive,negative}.wav` folders -- zip it on the laptop first, upload the zip, `!unzip` it here; thousands of loose small files upload far slower)

Checkpoints are read from `<drive_root>/data/checkpoints/` (Section 8/9's output) -- no separate upload needed if the heads were trained in this same Drive account. If they were trained/copied elsewhere, copy `rhythm_head.pt` / `melody_head.pt` / `timbre_head.pt` into that folder first (they're gitignored, so `git pull` will never fetch them).

In [ ]:
from pathlib import Path

from pipeline.config import PATHS  # already configured by Section 4/9 (configure_for_colab)

triplets_dir = drive_root / "triplets"
pool_path = triplets_dir / "pool_pilot.jsonl"
audio_dir = triplets_dir / "audio_pilot"
checkpoints_dir = PATHS.checkpoints_dir  # <drive_root>/data/checkpoints

assert pool_path.exists(), f"Upload pool_pilot.jsonl to {pool_path} first"
assert audio_dir.exists(), f"Upload + unzip audio_pilot/ to {audio_dir} first"
for factor in ("rhythm", "melody", "timbre"):
    ckpt = checkpoints_dir / f"{factor}_head.pt"
    assert ckpt.exists(), f"{ckpt} missing -- see the note above on copying checkpoints"
print("All inputs present.")

In [ ]:
import json

import torch
from demucs.pretrained import get_model

from eval.embed_triplets import embed_pool, load_heads
from pipeline.config import MODEL
from pipeline.mert_embed import MertEmbedder

device = "cuda" if torch.cuda.is_available() else "cpu"
demucs_model = get_model("htdemucs").to(device).eval()
embedder = MertEmbedder(MODEL.mert_model, device=device)

triplets = [json.loads(line) for line in pool_path.open() if line.strip()]
heads = load_heads(checkpoints_dir, {t["factor"] for t in triplets})

emb_rows, remapped = embed_pool(triplets, audio_dir, heads, demucs_model, embedder, device)
print(f"Embedded {len(emb_rows)} clips across {len(remapped)} triplets")

In [ ]:
import numpy as np
import pandas as pd

from eval.triplet_accuracy import triplet_accuracy

pd.DataFrame(emb_rows).to_parquet(triplets_dir / "embeddings_pilot.parquet")
with (triplets_dir / "pool_pilot_remapped.jsonl").open("w") as f:
    for t in remapped:
        f.write(json.dumps(t) + "\n")

embeddings = {row["track_id"]: np.asarray(row["embedding"]) for row in emb_rows}
summary = triplet_accuracy(remapped, embeddings)
for k, v in summary.items():
    print(f"{k}: {v:.4f}")

## Beyond this notebook

Deployment (Qdrant Cloud, API, frontend, Modal upload worker): see `DEPLOY.md`.
Paper-track triplet *generation* (real-audio stem manipulation, CPU-only) happens outside this notebook: `training/triplet_engine.py` + `training/triplet_render.py`, run on a laptop against the MUSDB18-HQ/MoisesDB zips. Section 12 above picks up from there (embed + machine baselines).